# Walkthrough — L1: General Linear Models & Basis Functions

**O que este walkthrough faz:** explica célula por célula o notebook `L1_General_Linear_Models_Basis_Functions.ipynb`, com foco nas ferramentas Python/NumPy/SciPy/Matplotlib usadas, por que cada escolha foi feita e como replicar os padrões.

**Conceitos matemáticos** já são explicados no notebook original — aqui o foco é no **código**.

---

## Roteiro
1. Setup e Configurações Globais
2. Lambda Functions e Subplots com `ax.annotate`
3. Design Matrix e Normal Equations
4. Polynomial Basis — `lstsq` vs `solve`
5. Gaussian RBF Basis — `np.column_stack` com list comprehension
6. Fourier Basis — `np.ones_like`
7. B-Splines — Cox-de Boor e `scipy.interpolate`
8. Pipeline Unificado — função genérica de ajuste

---
## 1. Setup e Configurações Globais

### O que o código faz

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import BSpline

np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
```

### Como cada ferramenta funciona

**`np.random.seed(42)`**  
Define o estado inicial do gerador de números aleatórios do NumPy. Qualquer número pode ser usado — o importante é que resultados sejam **reproduzíveis**: rodar o notebook duas vezes gera exatamente os mesmos dados aleatórios.

**`plt.rcParams`**  
Dicionário global de configurações do Matplotlib. Funciona como CSS para gráficos: define defaults que se aplicam a **todos os plots** no notebook.

| Chave | O que controla |
|-------|----------------|
| `"figure.figsize"` | Tamanho padrão da figura em polegadas `(largura, altura)` |
| `"font.size"` | Tamanho de fonte base para todos os textos |
| `"axes.grid"` | Liga/desliga grade em todos os eixos |
| `"grid.alpha"` | Transparência da grade (0=invisível, 1=sólida) |

**Por que `alpha=0.3` para a grade?**  
A grade ajuda a leitura mas não deve competir com os dados. Alpha baixo mantém a grade sutil.

---
## 2. Lambda Functions, Subplots e `ax.annotate`

### O que o código faz

```python
f_verdadeira = lambda x: np.sin(2 * np.pi * x)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
ax.plot([xi, xi], [f_verdadeira(xi), yi], 'r-', alpha=0.4, lw=1)
ax.annotate('...', xy=(0.3, f_verdadeira(0.3)), xytext=(0.05, -1.3),
            arrowprops=dict(arrowstyle='->', color='green'), ...)
```

### Lambda Functions

```python
f_verdadeira = lambda x: np.sin(2 * np.pi * x)
```

`lambda` cria uma **função anônima** em uma linha. É equivalente a:

```python
def f_verdadeira(x):
    return np.sin(2 * np.pi * x)
```

Útil para funções simples que serão usadas poucas vezes. Como `x` pode ser um array NumPy, `np.sin(2 * np.pi * x)` calcula o seno de cada elemento vetorialmente.

### `plt.subplots` — layout de múltiplos eixos

```python
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
```

- `1, 2` → 1 linha, 2 colunas de subplots  
- Retorna `(fig, axes)` onde `axes` é um array NumPy de eixos  
- `axes[0]` → subplot esquerdo, `axes[1]` → subplot direito  
- Para grades maiores (ex: `2, 3`), `axes` tem shape `(2, 3)` e acessa-se com `axes[linha, coluna]`

### Traçando segmentos verticais (resíduos)

```python
for xi, yi in zip(x_treino, y_treino):
    ax.plot([xi, xi], [f_verdadeira(xi), yi], 'r-', alpha=0.4, lw=1)
```

`ax.plot([x_start, x_end], [y_start, y_end])` traça uma linha entre dois pontos. Passando o mesmo valor de x duas vezes, o segmento é **vertical**. Isso representa visualmente o erro ε entre o valor observado e o valor real.

### `ax.annotate` — seta com texto

```python
ax.annotate('Inference: ...', 
            xy=(0.3, f_verdadeira(0.3)),   # ponto onde a seta aponta
            xytext=(0.05, -1.3),           # posição do texto
            arrowprops=dict(arrowstyle='->', color='green'))
```

- `xy` = coordenadas da **ponta da seta** (o ponto a ser anotado)  
- `xytext` = coordenadas do **texto** (de onde a seta parte)  
- `arrowprops` controla o estilo da seta: `'->'` é a mais comum

---
## 3. Design Matrix e Normal Equations

### O que o código faz

```python
X = np.column_stack([np.ones(n_obs), x_lin])
a_hat = np.linalg.solve(X.T @ X, X.T @ y_lin)
```

### `np.column_stack` — construindo a Design Matrix

```python
np.column_stack([np.ones(n_obs), x_lin])
```

Empilha arrays 1D como **colunas** de uma matriz 2D:
- Primeira coluna: `np.ones(n_obs)` — vetor de uns (para o intercept `a₀`)
- Segunda coluna: `x_lin` — os valores da feature

```
[[1.0, -1.8],
 [1.0,  0.3],
 [1.0,  1.7], ...]
```

Sem a coluna de uns, não há intercept e a reta seria forçada a passar pela origem.

**Alternativas equivalentes:**
```python
# np.vstack + .T
X = np.vstack([np.ones(n_obs), x_lin]).T

# np.c_ (açúcar sintático)
X = np.c_[np.ones(n_obs), x_lin]
```

### `np.linalg.solve` — resolvendo o sistema linear

```python
a_hat = np.linalg.solve(X.T @ X, X.T @ y_lin)
```

Resolve o sistema `(XᵀX) â = Xᵀy` para `â`.

**Por que `solve` e não `np.linalg.inv(X.T @ X) @ (X.T @ y)`?**

| Abordagem | Custo | Estabilidade | Recomendada? |
|-----------|-------|--------------|-------------|
| `solve(A, b)` | O(n³) | Alta — usa decomposição LU | ✅ Sim |
| `inv(A) @ b` | O(n³) + O(n²) | Baixa — propaga erros de arredondamento | ❌ Não |

`solve` nunca calcula a inversa explicitamente — resolve diretamente pelo sistema triangular após decomposição LU. Mais rápido e numericamente estável.

### O operador `@` vs `*`

```python
X.T @ X    # Multiplicação de matrizes: shape (2,N) @ (N,2) → (2,2)
X.T * X    # Multiplicação element-wise: ERRO se shapes incompatíveis
```

`@` é a multiplicação matricial (produto interno). `*` é element-wise. Para álgebra linear, sempre `@`.

### Print intermediário: `XᵀX` e `Xᵀy`

```python
print(X.T @ X)  # Gram matrix
print(X.T @ y_lin)
```

A **Gram matrix** `XᵀX` é sempre simétrica e positiva semi-definida. Sua condição (`np.linalg.cond(X.T @ X)`) indica estabilidade: número muito alto → sistema mal condicionado → `lstsq` é mais seguro que `solve`.

---
## 4. Polynomial Basis — `lstsq` vs `solve`

### O que o código faz

```python
def polynomial_basis_matrix(x, degree):
    return np.column_stack([x**m for m in range(degree + 1)])

B_treino = polynomial_basis_matrix(x_nl, grau)
a_hat = np.linalg.lstsq(B_treino, y_nl, rcond=None)[0]
B_plot = polynomial_basis_matrix(x_plot, grau)
y_pred = B_plot @ a_hat
```

### List comprehension para construir colunas

```python
np.column_stack([x**m for m in range(degree + 1)])
```

Para `degree=3` e `x = [0.1, 0.5, 0.9]`, isso cria:

```
[[0.1⁰, 0.1¹, 0.1², 0.1³],
 [0.5⁰, 0.5¹, 0.5², 0.5³],
 [0.9⁰, 0.9¹, 0.9², 0.9³]]
```

A primeira coluna é sempre 1 (qualquer número elevado a 0), absorvendo o intercept automaticamente.

### `np.linalg.lstsq` — mínimos quadrados robusto

```python
a_hat = np.linalg.lstsq(B_treino, y_nl, rcond=None)[0]
```

**Por que `lstsq` em vez de `solve` aqui?**

Para polinômios de grau alto, a matriz `B` pode ter colunas quase linearmente dependentes (valores de `x` próximos elevados a potências altas ficam muito similares). Isso torna `BᵀB` mal condicionada — `solve` produz resultados instáveis.

`lstsq` usa **SVD (Singular Value Decomposition)** internamente — trata matrizes singulares e quase-singulares com estabilidade.

**Por que `[0]`?** `lstsq` retorna uma tupla `(solução, resíduos, rank, valores_singulares)`. O `[0]` extrai apenas a solução.

**`rcond=None`**: usa a tolerância padrão moderna (evita o warning `FutureWarning` de versões antigas do NumPy).

### Comparação rápida

| Situação | Use |
|----------|-----|
| `BᵀB` bem condicionada (grau baixo, N >> p) | `np.linalg.solve(B.T@B, B.T@y)` |
| Grau alto, features correlacionadas, `p` próximo de `N` | `np.linalg.lstsq(B, y, rcond=None)[0]` |
| Sempre seguro (apenas mais lento) | `np.linalg.lstsq` |

### Predição: `B_plot @ a_hat`

```python
B_plot = polynomial_basis_matrix(x_plot, grau)  # avalia as basis nos novos pontos
y_pred = B_plot @ a_hat                          # combinação linear dos coeficientes
```

`x_plot` tem 300 pontos uniformes — a curva suave é criada avaliando o modelo em muitos pontos, não apenas nos dados de treino.

### Visualizando as próprias basis functions

```python
for m in range(grau + 1):
    plt.plot(x_vis, x_vis**m, label=f'$B_{m}(x) = x^{m}$')
```

Cada `x_vis**m` é o vetor de valores da m-ésima basis function em 200 pontos. Plotar isso revela por que graus altos causam problemas: as curvas `x^10`, `x^11` ficam quase idênticas → colunas de B quase iguais → sistema mal condicionado.

---
## 5. Gaussian RBF Basis

### O que o código faz

```python
def rbf_basis_matrix(x, centers, sigma):
    return np.column_stack([
        np.exp(-(x - c)**2 / (2 * sigma**2)) for c in centers
    ])

centers = np.linspace(-1, 1, n_centers)
B_treino = rbf_basis_matrix(x_nl, centers, sigma)
a_hat = np.linalg.lstsq(B_treino, y_nl, rcond=None)[0]
```

### Construção da matrix RBF

```python
np.column_stack([
    np.exp(-(x - c)**2 / (2 * sigma**2)) for c in centers
])
```

Para cada centro `c`, calcula a Gaussiana `exp(-(x-c)²/2σ²)` em todos os pontos de `x`. O resultado é um array 1D que vira uma coluna de `B`.

**Broadcasting:** `x` tem shape `(N,)` e `c` é um escalar. `(x - c)` funciona elemento a elemento por broadcasting. O NumPy subtrai `c` de cada elemento de `x`.

### Por que `np.linspace` para os centros?

```python
centers = np.linspace(-1, 1, n_centers)
```

Distribuir centros uniformemente é a estratégia mais simples. Alternativas:
- Usar os próprios pontos de treino como centros (K-nearest, Nystrom)
- Aprender os centros junto com os coeficientes (RBF Networks)

### Efeito de `sigma` — três plots lado a lado

```python
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, sig in zip(axes, sigmas):
    B_treino = rbf_basis_matrix(x_nl, centers, sig)
    ...
    ax.set_title(f'$\\sigma = {sig}$')
```

O padrão `zip(axes, sigmas)` itera em paralelo sobre os eixos e os valores de sigma — para cada par `(ax, sig)`, treina um modelo diferente e plota no subplot correspondente.

**Escape em f-strings LaTeX:** `f'$\\sigma = {sig}$'` — dentro de f-strings, `\\` produz o `\` que o LaTeX precisa. Uma barra invertida simples seria interpretada como escape Python.

---
## 6. Fourier Basis — `np.ones_like` e `np.concatenate`

### O que o código faz

```python
def fourier_basis_matrix(x, n_freqs, periodo=1.0):
    cols = [np.ones_like(x)]  # termo constante a0
    for k in range(1, n_freqs + 1):
        cols.append(np.sin(2 * np.pi * k * x / periodo))
        cols.append(np.cos(2 * np.pi * k * x / periodo))
    return np.column_stack(cols)
```

### `np.ones_like`

```python
np.ones_like(x)
```

Cria um array de uns com o **mesmo shape e dtype que `x`**. Equivalente a `np.ones(len(x))` para arrays 1D, mas mais seguro — se `x` mudar de tamanho, `ones_like` ainda funciona.

### Construção incremental com lista

O padrão aqui é diferente da Polynomial Basis:

```python
# Polynomial: list comprehension (todas as colunas de uma vez)
np.column_stack([x**m for m in range(degree + 1)])

# Fourier: lista mutável (adiciona colunas incrementalmente)
cols = [np.ones_like(x)]
for k in range(1, n_freqs + 1):
    cols.append(np.sin(...))  # coluna 2k-1
    cols.append(np.cos(...))  # coluna 2k
np.column_stack(cols)
```

Lista mutável é usada quando o número de colunas adicionadas por iteração é variável (aqui 2 por frequência). Ambas as abordagens são equivalentes em desempenho.

### Número de colunas resultante

Para `n_freqs=K`:
- 1 coluna constante (a₀)
- K colunas de seno
- K colunas de cosseno
- **Total: 2K + 1 colunas**

O notebook plota senos (azul) e cossenos (vermelho) com estilos de linha diferentes para distinguir visualmente as frequências.

### Subplots com `2, 1` (vertical)

```python
fig, axes = plt.subplots(2, 1, figsize=(11, 7))
```

2 linhas, 1 coluna → dois gráficos empilhados verticalmente. Para figuras altas que precisam de espaço horizontal para os dados.

---
## 7. B-Splines — Cox-de Boor e `scipy.interpolate`

### O que o código faz (parte 1 — implementação manual)

```python
def b_spline_cox_de_boor(x, knots, m, p):
    t = knots
    if p == 0:
        return np.where((t[m] <= x) & (x < t[m + 1]), 1.0, 0.0)
    
    left = np.zeros_like(x, dtype=float)
    right = np.zeros_like(x, dtype=float)
    
    denom_l = t[m + p] - t[m]
    if denom_l > 0:
        left = (x - t[m]) / denom_l * b_spline_cox_de_boor(x, knots, m, p - 1)
    ...
    return left + right
```

### `np.where` — condicional vetorial

```python
np.where((t[m] <= x) & (x < t[m + 1]), 1.0, 0.0)
```

`np.where(condição, valor_se_true, valor_se_false)` aplica a condição elemento a elemento:
- Para pontos `x` dentro do intervalo `[t[m], t[m+1])` → retorna `1.0`
- Para pontos fora → retorna `0.0`

O operador `&` é o AND bit a bit — usado em arrays booleanos NumPy. `and` (Python nativo) não funciona com arrays.

### `np.zeros_like` — inicializando acumuladores

```python
left = np.zeros_like(x, dtype=float)
```

Cria array de zeros com mesmo shape que `x`, mas forçando `dtype=float` (mesmo que `x` seja inteiro). Necessário para inicializar o acumulador antes de modificá-lo condicionalmente.

### Recursão e divisão por zero

```python
denom_l = t[m + p] - t[m]
if denom_l > 0:
    left = ...
```

Knots repetidos causam `denom = 0` — divisão por zero. A convenção Cox-de Boor define `0/0 = 0`, então simplesmente não calculamos o termo quando o denominador é zero (`left` permanece zero).

### Clamped B-Splines — knots nas bordas

```python
grau = 3
knots_internos = np.linspace(0, 1, 6)
knots = np.concatenate([[0] * grau, knots_internos, [1] * grau])
```

**Por que repetir as bordas?** Sem repetição, a B-Spline não passa pelos pontos extremos — flutua dentro do intervalo. Com `p` repetições nas bordas, a spline fica "presa" (`clamped`) nos extremos.

```
knots = [0, 0, 0, 0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1, 1]
                         ↑ internos ↑              ↑ clamped
```

**Número de basis functions:** `n_basis = len(knots) - grau - 1`

### `x_vis = np.linspace(0, 1 - 1e-9, 500)` — por que `1 - 1e-9`?

A última B-Spline é definida no intervalo `[t[m], t[m+1])` com **parêntese aberto** à direita. Se `x = 1.0` exato, nenhuma basis cobre esse ponto (pois a última comparação é `x < t[n]`). Usar `1 - 1e-9` evita esse edge case numérico.

### Parte 2 — `scipy.interpolate.make_lsq_spline`

```python
from scipy.interpolate import make_lsq_spline

knots_fit = np.linspace(0.1, 0.9, 8)
spl = make_lsq_spline(x_cs, y_cs, knots_fit, k=3)
y_pred = spl(x_spl)
```

`make_lsq_spline` ajusta uma B-Spline aos dados por mínimos quadrados:
- `x_cs`, `y_cs`: dados de treino (x deve ser ordenado)  
- `knots_fit`: knots internos (não inclua as bordas — `make_lsq_spline` adiciona automaticamente)  
- `k=3`: grau cúbico  
- Retorna um objeto `BSpline` que é chamável: `spl(x_novo)` faz predição

### Compact Support — modificando um coeficiente

```python
c_modificado = spl.c.copy()          # copia os coeficientes
idx_central = len(c_modificado) // 2
c_modificado[idx_central] += 1.0     # perturba apenas um
spl_modificado = BSpline(spl.t, c_modificado, spl.k)  # recria com novos coefs
```

- `spl.c` — coeficientes (`a_hat`)
- `spl.t` — sequência de knots
- `spl.k` — grau
- `.copy()` — essencial para não modificar o objeto original (arrays são mutáveis)

### `ax.axvspan` — região destacada

```python
ax.axvspan(spl.t[idx_central], spl.t[idx_central + spl.k + 1], 
           alpha=0.1, color='red', label='Região afetada')
```

`axvspan(x_min, x_max)` preenche uma região vertical inteira entre dois valores de x. Aqui mostra visualmente o suporte compacto — apenas essa faixa do gráfico muda quando perturbamos o coeficiente central.

---
## 8. Pipeline Unificado — função genérica de ajuste

### O que o código faz

```python
def ajustar_e_prever(B_treino, B_pred, y, lam=0.0):
    BTB = B_treino.T @ B_treino
    BTy = B_treino.T @ y
    if lam > 0:
        BTB += lam * np.eye(BTB.shape[0])
    a_hat = np.linalg.solve(BTB, BTy)
    return B_pred @ a_hat
```

### Design pattern: separar B_treino e B_pred

A função recebe as matrizes B já construídas — não os dados `x`, `y`. Isso separa:
1. **Construção de features:** cada tipo de basis tem sua própria função (`rbf_basis_matrix`, etc.)
2. **Ajuste:** esta função é sempre a mesma — muda apenas o `B`

Isso é o mesmo princípio do sklearn `Pipeline`: separar transformação de estimação.

### Ridge opcional com `lam=0.0`

```python
if lam > 0:
    BTB += lam * np.eye(BTB.shape[0])
```

`np.eye(n)` cria a matriz identidade n×n. Adicionar `λI` à diagonal de `BᵀB` é a Ridge Regression — estabiliza o sistema e regulariza os coeficientes.

**`BTB.shape[0]`** extrai o número de linhas (= número de colunas, pois `BᵀB` é quadrada = número de basis functions).

### `+=` modifica in-place

```python
BTB += lam * np.eye(BTB.shape[0])
```

Equivalente a `BTB = BTB + lam * np.eye(...)`, mas sem criar um novo array. Para matrizes grandes, economiza memória.

### Plotagem com `constrained_layout`

```python
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
```

Para figuras com subplots complexos, adicionar `constrained_layout=True` evita que colorbars e títulos se sobreponham. No notebook original não foi necessário pois não há colorbars aqui — mas é boa prática.

### Print final de dimensões

```python
print(f"{'Basis':<30} {'Shape de B':<20} {'# coeficientes â'}")
print(f"{'Polynomial (grau 6)':<30} {str(B_poly_tr.shape):<20} {B_poly_tr.shape[1]}")
```

**f-string com alinhamento:** `:<30` alinha à esquerda com 30 caracteres. Cria tabelas alinhadas no terminal sem importar `tabulate`.

---

## Resumo das Ferramentas

| Ferramenta | Uso principal no notebook | Alternativa |
|-----------|--------------------------|-------------|
| `lambda x: ...` | Funções matemáticas simples | `def f(x): return ...` |
| `np.column_stack([...])` | Montar Design Matrix | `np.c_[...]`, `np.vstack().T` |
| `np.linalg.solve(A, b)` | Normal Equations quando BᵀB bem condicionada | — |
| `np.linalg.lstsq(B, y)[0]` | Quando B mal condicionada (grau alto) | — |
| `np.where(cond, v1, v2)` | Condicional vetorial (sem loop) | `np.select` para múltiplas condições |
| `np.concatenate` | Juntar arrays 1D | `np.append` (mais lento para múltiplos) |
| `np.zeros_like(x, dtype=float)` | Acumulador com mesmo shape | `np.zeros(len(x))` |
| `make_lsq_spline(x, y, knots, k)` | Ajuste de B-Spline por LS | Manual com Cox-de Boor |
| `BSpline(t, c, k)` | Criar B-Spline com coefs customizados | — |
| `ax.axvspan(x1, x2)` | Destacar região horizontal | `ax.fill_betweenx` |
| `ax.annotate` | Setas com texto | `ax.text` (sem seta) |

---

## Como Proceder ao Usar Estes Padrões

1. **Para qualquer Basis Function nova:** crie uma função `my_basis_matrix(x, params)` que retorna `B` com shape `(N, M)`. Passe para `ajustar_e_prever`.

2. **Para diagnosticar instabilidade numérica:** verifique `np.linalg.cond(B.T @ B)`. Se > 1e10, use `lstsq` ou adicione regularização (`lam > 0`).

3. **Para B-Splines com scipy:** sempre verifique que `x` está ordenado (`np.sort`) antes de chamar `make_lsq_spline`.

4. **Para Compact Support:** modifique `spl.c.copy()` — nunca `spl.c` diretamente (os arrays scipy são às vezes somente leitura).